In [1]:
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "text.usetex": True,
    "pgf.texsystem": "lualatex",
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [2]:
with open('data/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'],
    msq.Component.SIG : xsec['gg4l_sig'],
    msq.Component.INT : xsec['gg4l_int'],
    msq.Component.BKG : xsec['gg4l_bkg']
}

filenames = {
    msq.Component.SBI : 'data/ggZZ2e2m_sbi/events.csv',
    msq.Component.SIG : 'data/ggZZ2e2m_sig/events.csv',
    msq.Component.INT : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.BKG : 'data/ggZZ2e2m_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.SIG : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [3]:
events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows=10000)
events_sbi = zz4l.analyze(events_sbi).shuffle()
events_sig = events_sbi.reweight(msq.Component.SBI, msq.Component.SIG)
events_int = events_sbi.reweight(msq.Component.SBI, msq.Component.INT)

In [14]:
c6_vals = np.linspace(-25, 25, 101)
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

c6_pts_sig = np.array([-10,-5,0, 5,10])
c6_pts_int = np.array([-10,   0,   10])
c6_pts_sbi = np.array([-10,-5,0, 5,10])

mod_sig_c6 = c6.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_values=c6_pts_sig)
mod_int_c6 = c6.Modifier(baseline=msq.Component.INT, events=events_int, c6_values=c6_pts_int)
mod_sbi_c6 = c6.Modifier(baseline=msq.Component.SBI, events=events_sbi, c6_values=c6_pts_sbi)

wt_sig_c6, prob_sig_c6 = mod_sig_c6.modify(c6_vals)
wt_int_c6, prob_int_c6 = mod_int_c6.modify(c6_vals)
wt_sbi_c6, prob_sbi_c6 = mod_sbi_c6.modify(c6_vals)

In [15]:
sig_bsm_over_sm = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis]
int_bsm_over_sm = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis]
sbi_bsm_over_sm = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis]

In [16]:
event_index = 4321

In [ ]:
fig, (ax_sig, ax_int, ax_sbi) = plt.subplots(3,1, gridspec_kw={'height_ratios': [1,1,1]}, figsize=(5,5), sharex=True)

# ax_sig.plot(c6_vals, sig_bsm_over_sm[90], linestyle='--')
# ax_int.plot(c6_vals, int_bsm_over_sm[90], linestyle='--')
# ax_sbi.plot(c6_vals, sbi_bsm_over_sm[90], linestyle='--')

msq_sig_c6 = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis] * events_sig.components['msq_sig_sm'].to_numpy()[:,np.newaxis]
msq_int_c6 = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis] * events_int.components['msq_int_sm'].to_numpy()[:,np.newaxis]
msq_sbi_c6 = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis] * events_sbi.components['msq_sbi_sm'].to_numpy()[:,np.newaxis]

c6_pts_all = []
msq_sig_pts = []
msq_int_pts = []
msq_sbi_pts = []
for c6_val, msq_name in mcfm.mcfm_component_c6[msq.Component.SBI].items():
    c6_pts_all.append(c6_val)
    msq_sbi_pts.append(events_sbi.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.SIG].values():
    msq_sig_pts.append(events_sig.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.INT].values():
    msq_int_pts.append(events_int.components[msq_name][event_index])
msq_sig_pts = np.array(msq_sig_pts)
msq_int_pts = np.array(msq_int_pts)
msq_sbi_pts = np.array(msq_sbi_pts)

ax_sig.plot(c6_vals, msq_sig_c6[event_index], linestyle='--', color='blue')
ax_int.plot(c6_vals, msq_int_c6[event_index], linestyle='--', color='blue')
ax_sbi.plot(c6_vals, msq_sbi_c6[event_index], linestyle='--', color='blue')

ax_sig.scatter(c6_pts_all, msq_sig_pts, color='black')
ax_int.scatter(c6_pts_all, msq_int_pts, color='black')
ax_sbi.scatter(c6_pts_all, msq_sbi_pts, color='black')

ax_sig.scatter(c6_pts_sig, msq_sig_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_sig))[0]], color='blue')
ax_int.scatter(c6_pts_int, msq_int_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_int))[0]], color='blue')
ax_sbi.scatter(c6_pts_sbi, msq_sbi_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_sbi))[0]], color='blue')

ax_sig.set_ylabel('$|\\mathcal{M}_{gg \\rightarrow h^{\\ast} \\rightarrow ZZ}|^2$')
ax_int.set_ylabel('$2\\mathrm{Re}(\\mathcal{M}^{\\dag}_{gg \\rightarrow h^{\\ast} \\rightarrow ZZ} \\mathcal{M}_{gg \\rightarrow ZZ})$')
ax_sbi.set_ylabel('$|\\mathcal{M}_{gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ}|^2$')

ax_sbi.set_xlabel('$c_6$')

plt.tight_layout()
plt.savefig('msq_bsm_over_sm.pdf')
plt.close()

[3.067150e-11 1.418922e-12 5.268779e-12 2.305460e-11 4.077761e-11
 4.960668e-11 4.587836e-11 3.109689e-11 1.193415e-11 2.297256e-13
 1.299085e-11]
[False False False  True  True  True  True  True False False False]
